In [28]:
import requests
from bs4 import BeautifulSoup
import time

# ---------------- CONFIGURATION ----------------
BASE_URL_MAIN = "https://www.sanjivanicoe.org.in"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

DEPARTMENTS = {
    "Computer Engineering": "computer-engineering",
    "Information Technology": "information-technology",
    "Civil Engineering": "civil-engineering",
    "Mechanical Engineering": "mechanical-engineering",
    "Electrical Engineering": "electrical-engineering",
    "E&TC Engineering": "e-tc-engineering",
    "Structural Engineering": "structural-engineering",
    "Mechatronics Engineering": "mechatronics-engineering",
    "MBA": "mba"
}

# ---------------- HELPER FUNCTIONS ----------------

def get_soup(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        if response.status_code == 200:
            return BeautifulSoup(response.text, "html.parser")
    except Exception:
        pass
    return None

def clean_text(text):
    if not text: return ""
    return " ".join(text.split())

# ---------------- 1. DIRECTOR (YOUR CODE) ----------------
def get_director_name():
    url = f"{BASE_URL_MAIN}/index.php/about-us/principal-s-desk"
    soup = get_soup(url)
    
    if soup:
        # --- USING YOUR LOGIC HERE ---
        # Look through specific tags and take the FIRST one with Dr./Prof.
        for tag in soup.find_all(["h3", "h4", "strong", "p"]):
            text = tag.get_text(strip=True)
            if "Dr." in text or "Prof." in text:
                return text # Return immediately when found
                
    return "Director Name Not Found"

# ---------------- 2. TRUSTEES ----------------
def get_trustees():
    url = f"{BASE_URL_MAIN}/index.php/about-us/board-of-trustees"
    soup = get_soup(url)
    trustees_list = []
    
    if soup:
        for tr in soup.find_all("tr"):
            cells = tr.find_all("td")
            if len(cells) >= 2:
                name = clean_text(cells[0].get_text())
                role = clean_text(cells[1].get_text())
                if name and role:
                    trustees_list.append(f"{name} — {role}")
    return trustees_list

# ---------------- 3. DEPARTMENT DATA ----------------
def get_dept_data(dept_slug):
    # --- GET HOD ---
    hod_url = f"{BASE_URL_MAIN}/index.php/department/{dept_slug}/hod-s-desk"
    hod_soup = get_soup(hod_url)
    hod_name = "HOD Not Found"
    
    if hod_soup:
        for tag in hod_soup.find_all(["h3", "h4", "strong", "b", "p"]):
            text = clean_text(tag.get_text())
            if ("Dr." in text or "Prof." in text) and len(text) < 50:
                if "," not in text and "Welcome" not in text:
                    hod_name = text
                    break # Take first match
    
    # --- GET FACULTY ---
    fac_url = f"{BASE_URL_MAIN}/index.php/department/{dept_slug}/faculty"
    fac_soup = get_soup(fac_url)
    faculty_list = set()
    
    if fac_soup:
        tags = fac_soup.find_all(["h3", "h4", "strong", "span"])
        for tag in tags:
            text = clean_text(tag.get_text())
            if any(x in text for x in ["Dr.", "Prof.", "Mr.", "Mrs."]):
                if 4 < len(text) < 50:
                    faculty_list.add(text)
                    
    return hod_name, sorted(list(faculty_list))

# ---------------- MAIN EXECUTION ----------------

output_data = [] 

print("🚀 STARTING SANJIVANI MASTER SCRAPER...\n")

# --- PART 1: College & Director ---
director = get_director_name()
print(f"🏛️  COLLEGE: Sanjivani College of Engineering")
print(f"👤 DIRECTOR: {director}")

output_data.append("SANJIVANI COLLEGE OF ENGINEERING")
output_data.append("="*40)
output_data.append(f"Director: {director}")
output_data.append("="*40)

# --- PART 2: Trustees ---
print("\n--- 🤝 BOARD OF TRUSTEES ---")
trustees = get_trustees()
output_data.append("\nBOARD OF TRUSTEES:")
if trustees:
    for t in trustees:
        print(f"  - {t}")
        output_data.append(f"  - {t}")
else:
    output_data.append("  No trustees found.")
output_data.append("-" * 40)

# --- PART 3: Departments ---
print("\n--- 📚 DEPARTMENTS & FACULTY ---")
output_data.append("\nDEPARTMENT DETAILS:")

for dept_name, slug in DEPARTMENTS.items():
    print(f"\n🔵 Processing: {dept_name}...")
    
    hod, faculty = get_dept_data(slug)
    
    print(f"   👤 HOD: {hod}")
    print(f"   👥 Faculty Found: {len(faculty)}")
    
    output_data.append(f"\nDEPARTMENT: {dept_name}")
    output_data.append(f"HOD: {hod}")
    output_data.append("FACULTY LIST:")
    if faculty:
        for i, f in enumerate(faculty, 1):
            output_data.append(f"  {i}. {f}")
    else:
        output_data.append("  No faculty found.")
    output_data.append("-" * 30)
    
    time.sleep(1) 

# --- PART 4: Save ---
filename = "sanjivani_full_report.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write("\n".join(output_data))

print(f"\n✔ SUCCESS! Report saved to '{filename}'")

🚀 STARTING SANJIVANI MASTER SCRAPER...

🏛️  COLLEGE: Sanjivani College of Engineering
👤 DIRECTOR: Dr. R. A. Kapgate

--- 🤝 BOARD OF TRUSTEES ---
  - 01 — Hon’ble Shri Nitinrao Shankarrao Kolhe
  - 02 — Hon’ble Shri Amit Nitinrao Kolhe
  - 03 — Hon’ble Shri Bipindada Shankarrao Kolhe
  - 04 — Hon’ble Shri Jairam Vishram Gadakh
  - 05 — Hon’ble Shri Shivjirao Anandrao Sandhan
  - 06 — Hon’ble Shri Sumit Nitinrao Kolhe
  - 07 — Hon’ble Shri Vivek Bipindada Kolhe
  - 08 — Hon’ble Shri A.D. Antre

--- 📚 DEPARTMENTS & FACULTY ---

🔵 Processing: Computer Engineering...
   👤 HOD: Prof. Dr. Madhuri A.Jawale
   👥 Faculty Found: 23

🔵 Processing: Information Technology...
   👤 HOD: Prof. Dr. Madhuri A.Jawale
   👥 Faculty Found: 20

🔵 Processing: Civil Engineering...
   👤 HOD: Prof. Dr. A. S. Sayyad
   👥 Faculty Found: 19

🔵 Processing: Mechanical Engineering...
   👤 HOD: Dr. P.M.Patare
   👥 Faculty Found: 25

🔵 Processing: Electrical Engineering...
   👤 HOD: HOD Not Found
   👥 Faculty Found: 11

